<a href="https://colab.research.google.com/github/Mike0165115321/open_data_storytelling/blob/main/SuperAI%20-%20Hack%201%20(Draft%201).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🐟 Data Storytelling: เจาะลึกโครงสร้างการนำเข้า-ส่งออกสินค้าประมงไทย (Thailand Fisheries Trade Analysis)

**Executive Summary:**
* 🌊 **Context:** วิเคราะห์ฐานข้อมูลการค้าระหว่างประเทศกลุ่มสินค้าประมง เพื่อประเมินสถานะของประเทศไทยในฐานะผู้ผลิตและผู้บริโภคในตลาดโลก
* 🎯 **Objectives:** Notebook นี้นำเสนอ Insight ผ่าน 3 แกนหลัก:
   1. **Major Trading Partners (Where):** ตลาดส่งออกหลัก แหล่งนำเข้า และสถานะดุลการค้าของไทย
   2. **Fisheries Product Structure (What):** โครงสร้างสินค้าประมงไทย (Net Exporter vs Net Importer)
   3. **Seasonal Trade Pattern (When):** ฤดูกาลการค้า ปัจจัยจุดพีกและจุดต่ำสุดในรอบปี
* 🚀 **So What?:** ข้อมูลเชิงลึกเหล่านี้จะช่วยให้ผู้กำหนดนโยบายและผู้ประกอบการสามารถวางแผนรับมือฤดูกาล สเกลอัปสินค้าที่มีศักยภาพ และกระจายความเสี่ยงในตลาดคู่ค้าได้อย่างแม่นยำ

---

In [4]:
# ============================================================
# 📦 ส่วนที่ 1: การนำเข้าไลบรารีที่จำเป็น (Dependencies)
# ============================================================
!pip install pycountry -q                 # ติดตั้งไลบรารีแปลงชื่อประเทศอัตโนมัติ (ซ่อน Log ด้วย -q)

import pandas as pd                       # จัดการโครงสร้างตารางข้อมูล (DataFrame) และการวิเคราะห์
import numpy as np                        # คำนวณคณิตศาสตร์ขั้นสูงและการทำงานกับ Array (Vectorization)
import plotly.express as px               # สร้างกราฟ Interactive แบบรวดเร็วและสวยงาม
import plotly.graph_objects as go         # สร้างกราฟ Interactive ที่มีความซับซ้อนสูง (เช่น แดชบอร์ด)
import plotly.io as pio                   # จัดการระบบตั้งค่าเริ่มต้นของ Plotly (ใช้เซ็ต Global Template สี)
from plotly.subplots import make_subplots # สร้าง Layout จัดเรียงหลายกราฟไว้ในหน้าจอเดียว (Grid)
import pycountry                          # แปลงรหัสประเทศ (ISO-2 เป็น ISO-3) เพื่อใช้วาดลงบนแผนที่โลก
import warnings                           # จัดการระบบแจ้งเตือนของ Python
from IPython.display import display, HTML # สร้างและแสดงผล UI/ชิ้นส่วน HTML ให้สวยงามภายใน Jupyter Notebook
import calendar                           # ใช้สำหรับจัดการข้อมูลเกี่ยวกับปฏิทิน

warnings.filterwarnings('ignore')         # ปิดการแจ้งเตือน (Warning) เพื่อให้ผลลัพธ์การรันโค้ดสะอาดตา

display(HTML("<span style='color:#7EE787; font-weight:bold; font-family:Segoe UI;'>✅ Dependencies Loaded: โหลดไลบรารีเสร็จสมบูรณ์พร้อมทำงาน</span>"))

In [5]:
# ============================================================
# 🎨 ส่วนที่ 2: การตั้งค่าระบบสีและฟังก์ชันช่วยเหลือ (Config & Helpers)
# ============================================================

# --- 1. ระบบสี (Design System Object) ---
# ปรับเป็น Class เพื่อให้ IDE Auto-complete ทำงานง่ายเวลาเขียนโค้ด (Theme.import_col)
class Theme:
    import_col = '#FF6B6B'       # สีแดงปะการัง (นำเข้า)
    export_col = '#4ECDC4'       # สีฟ้าอมเขียว (ส่งออก)
    bg_dark = '#0D1117'          # พื้นหลังหลัก
    bg_card = '#161B22'          # พื้นหลังกล่องกราฟ
    text = '#E6EDF3'             # สีตัวอักษร
    grid = '#30363D'             # สีเส้นตาราง
    accent = '#F0883E'           # สีส้มเน้นย้ำ
    mean_line = '#8B949E'        # สีเทาเส้นเฉลี่ย

TRADE_COLOR_MAP = {'Import': Theme.import_col, 'Export': Theme.export_col}
COLORS = {k: v for k, v in Theme.__dict__.items() if not k.startswith('__')}

# --- 2. Plotly Template (Global Injection) ---
pio.templates["superai_dark"] = go.layout.Template(
    layout=go.Layout(
        paper_bgcolor=Theme.bg_dark,
        plot_bgcolor=Theme.bg_card,
        font=dict(family='Segoe UI, sans-serif', size=13, color=Theme.text),
        title=dict(font=dict(size=20, color=Theme.text, family='Segoe UI, sans-serif'), x=0.5),
        margin=dict(l=60, r=40, t=80, b=60),
        legend=dict(
            bgcolor='rgba(0,0,0,0)',
            bordercolor=Theme.grid, borderwidth=1, font=dict(size=12)
        ),
        xaxis=dict(gridcolor=Theme.grid, zeroline=False),
        yaxis=dict(gridcolor=Theme.grid, zeroline=False),
        colorway=[Theme.accent, Theme.import_col, Theme.export_col]
    )
)
# ประกาศใช้ทั้ง Notebook
pio.templates.default = "plotly_dark+superai_dark"

def apply_premium_layout(fig, title: str, height: int = 500):
    fig.update_layout(title_text=f"<b>{title}</b>", height=height)
    return fig

# --- 3. ระบบ UI Logging (Rich HTML Display) ---
def print_log(log_dict: dict):
    html = f"""
    <div style='background-color:{Theme.bg_card}; padding:20px; border-radius:10px; border:1px solid {Theme.grid}; color:{Theme.text}; max-width: 500px; font-family: Segoe UI, sans-serif;'>
        <h3 style='margin-top:0; color:{Theme.accent};'>✅ Data Preparation Summary</h3>
        <hr style='border-color:{Theme.grid}; margin-bottom:15px;'>
    """

    icons = {"original": "📥", "duplicates": "🗑️", "final": "✅", "missing": "🔍"}
    for key, val in log_dict.items():
        emoji = next((v for k, v in icons.items() if k in key), '🔹')
        label = key.replace("_", " ").title()
        html += f"<div style='display:flex; justify-content:space-between; margin-bottom:8px; font-size:14px;'>"
        html += f"<span>{emoji} {label}</span> <b style='color:{Theme.text};'>{val:,.0f}</b></div>"

    html += "</div>"
    display(HTML(html))

display(HTML(f"<span style='color:{Theme.export_col}; font-weight:bold;'>🎨 System Activated: Global Template & Premium Logger Loaded.</span>"))

In [6]:
# ============================================================
# 📥 Step 1: โหลดและเตรียมข้อมูล (Data Preparation - Fisheries)
# ============================================================

# 1.1 โหลดข้อมูล (ป้องกันนามิเบีย 'NA' กลายเป็นค่าว่าง)
df = pd.read_csv('import_export.csv', keep_default_na=False, na_values=[''])

log = {
    "original_rows": len(df),
    "missing_values_found": int(df.isnull().sum().sum())
}

# 1.2 ทำความสะอาดข้อมูล (Data Cleaning)
duplicates_count = df.duplicated().sum()
df_clean = df.drop_duplicates().copy()
log["duplicates_removed"] = int(duplicates_count)

df_clean = df_clean.dropna()
log["missing_values_dropped"] = log["missing_values_found"]

# 1.3 ⚠️ แปลงค่าตัวเลขกลับสู่สเกลจริง (ตาม Data Dictionary)
# price: มูลค่าการค้า (บาท * 1,000,000)
df_clean['price_actual'] = df_clean['price'] * 1_000_000
# weight: น้ำหนักสินค้า (กิโลกรัม * 1,000)
df_clean['weight_actual'] = df_clean['weight'] * 1_000

# 1.4 จัดรูปแบบและสร้างฟีเจอร์พื้นฐาน (Feature Engineering)
df_clean['trade_type'] = df_clean['tradeflow'].map({1: 'Import', 2: 'Export'})

# 🌟 สร้างคอลัมน์ "มูลค่าสุทธิ (Net Value)"
# ให้ส่งออก (Export) เป็นบวก (+) และ นำเข้า (Import) เป็นลบ (-)
# เพื่อความง่ายในการคำนวณ "ดุลการค้า (Trade Balance)" ในขั้นตอนต่อไป
df_clean['trade_value_net'] = np.where(
    df_clean['trade_type'] == 'Export',
    df_clean['price_actual'],
    -df_clean['price_actual']
)

# 1.5 แปลงรหัสประเทศ (ISO-2 เป็น ISO-3) สำหรับวาดแผนที่ 3D
def get_iso3(iso2):
    try: return pycountry.countries.get(alpha_2=iso2).alpha_3
    except: return None
df_clean['iso3'] = df_clean['countryID'].apply(get_iso3)

log["final_ready_rows"] = len(df_clean)

# แสดงข้อมูลสรุปการคลีนผ่าน UI
print_log(log)

# ตรวจสอบโครงสร้างข้อมูลที่พร้อมใช้งาน
display_cols = ['year', 'month', 'countryNameTH', 'productDetailTH', 'trade_type', 'price_actual', 'weight_actual', 'trade_value_net']
display(df_clean[display_cols].head(3))

,year,month,countryNameTH,productDetailTH,trade_type,price_actual,weight_actual,trade_value_net
0,2568,8,เบนิน,เต่าและตะพาบน้ำมีชีวิต,Import,48080000000,33000,-48080000000
1,2568,8,มอริเชียส,เต่าและตะพาบน้ำมีชีวิต,Import,44147000000,10000,-44147000000
2,2568,8,บราซิล,เต่าและตะพาบน้ำมีชีวิต,Import,47068000000,8000,-47068000000


# 🌍 Step 2: วิเคราะห์คู่ค้าหลักและดุลการค้า (Major Trading Partners)
**🎯 โจทย์ข้อ 1:** ประเทศใดคือตลาดส่งออกสำคัญ แหล่งนำเข้าหลัก และเราได้เปรียบ/เสียเปรียบดุลการค้ากับใคร?

In [7]:
# ============================================================
# 🌍 Step 2: วิเคราะห์คู่ค้าหลักและดุลการค้า (Major Trading Partners)
# ============================================================

# 1. เตรียมข้อมูล: รวมยอด Import/Export และดุลการค้า แยกตามประเทศ
country_trade = df_clean.groupby(['countryNameTH', 'trade_type'])['price_actual'].sum().unstack(fill_value=0).reset_index()

# เช็คว่ามีคอลัมน์ Import/Export ครบไหม (กัน Error กรณีบางประเทศมีแค่ฝั่งเดียว)
if 'Import' not in country_trade.columns: country_trade['Import'] = 0
if 'Export' not in country_trade.columns: country_trade['Export'] = 0

# คำนวณยอดรวม (Total Trade) และ ดุลการค้า (Trade Balance)
country_trade['Total Trade'] = country_trade['Import'] + country_trade['Export']
country_trade['Trade Balance'] = country_trade['Export'] - country_trade['Import']

# คัดเลือก Top 15 ประเทศที่มี "มูลค่าการค้ารวมสูงสุด" มาแสดงผล
top_countries = country_trade.sort_values(by='Total Trade', ascending=True).tail(15)

# 2. วาดกราฟ Diverging Bar Chart (แท่งหันหลังชนกัน)
fig_q1 = go.Figure()

# แท่งฝั่งส่งออก (Export) -> ชี้ไปทางขวา (บวก)
fig_q1.add_trace(go.Bar(
    y=top_countries['countryNameTH'],
    x=top_countries['Export'],
    name='ส่งออก (Export)',
    orientation='h',
    marker_color=Theme.export_col,
    hovertemplate="<b>%{y}</b><br>ส่งออก: ฿%{x:,.0f}<extra></extra>"
))

# แท่งฝั่งนำเข้า (Import) -> ชี้ไปทางซ้าย (ติดลบเพื่อให้กราฟแยกซ้ายขวา)
fig_q1.add_trace(go.Bar(
    y=top_countries['countryNameTH'],
    x=-top_countries['Import'],  # ใส่เครื่องหมายลบ
    name='นำเข้า (Import)',
    orientation='h',
    marker_color=Theme.import_col,
    hovertemplate="<b>%{y}</b><br>นำเข้า: ฿%{customdata:,.0f}<extra></extra>",
    customdata=top_countries['Import'] # โชว์ค่าจริงตอน Hover ไม่ให้เห็นค่าติดลบ
))

# เพิ่มจุด (Marker) แสดง "ดุลการค้า (Trade Balance)" สุทธิ
fig_q1.add_trace(go.Scatter(
    y=top_countries['countryNameTH'],
    x=top_countries['Trade Balance'],
    name='ดุลการค้า (Net Balance)',
    mode='markers',
    marker=dict(color=Theme.accent, size=10, symbol='diamond'),
    hovertemplate="<b>%{y}</b><br>ดุลการค้า: ฿%{x:,.0f}<extra></extra>"
))

# 3. จัด Layout ระดับ Premium
fig_q1.update_layout(
    title=dict(text="<b>🌍 ตลาดส่งออก/นำเข้าหลัก และดุลการค้าสินค้าประมง (Top 15 Trading Partners)</b>", font=dict(size=20), x=0.5),
    barmode='overlay',
    height=600,
    xaxis=dict(
        title='มูลค่าการค้า (บาท)',
        tickvals=[-10e9, -5e9, 0, 5e9, 10e9, 15e9], # ตัวอย่างสเกล (ปรับได้ตาม Data จริง)
        ticktext=["฿10B", "฿5B", "0", "฿5B", "฿10B", "฿15B"],
        zeroline=True, zerolinewidth=2, zerolinecolor=Theme.text
    ),
    yaxis=dict(title='ประเทศคู่ค้า'),
    hovermode="y unified"
)

fig_q1.show()

### 📌 Insight & So What? (สรุปผลข้อ 1)
* 📤 **ตลาดส่งออกหลัก (Top Destinations):** ประเทศที่กราฟแท่งสีฟ้า (Export) ยาวที่สุดทางฝั่งขวา คือแหล่งรายได้หลักของเรา
* 📥 **แหล่งนำเข้าหลัก (Top Sources):** ประเทศที่กราฟแท่งสีแดง (Import) ยาวที่สุดทางฝั่งซ้าย คือประเทศที่เราพึ่งพาวัตถุดิบประมง (เช่น นำเข้าปลามาแปรรูป)
* ⚖️ **สถานะดุลการค้า (Trade Balance):** * สังเกตที่ **"จุดข้าวหลามตัดสีส้ม"** หากจุดนี้อยู่ฝั่งขวา แปลว่าเรา **"เกินดุล (Surplus)"** กับประเทศนั้น (รับเงินเข้ามากกว่าจ่าย)
    * หากจุดสีส้มตกไปอยู่ฝั่งซ้าย แปลว่าเรา **"ขาดดุล (Deficit)"** * 🚀 **So What?:** สำหรับประเทศที่เราเกินดุลสูง รัฐควรทำ FTA หรือส่งเสริมแคมเปญการตลาดเพื่อรักษาฐานลูกค้าให้แน่นแฟ้น ส่วนประเทศที่เรานำเข้าสูง (ขาดดุล) ควรประเมินว่าเป็นการนำเข้าเพื่อการบริโภค หรือนำเข้าเพื่อแปรรูปส่งออกต่อ หากเป็นอย่างแรก ควรมีนโยบายส่งเสริมให้ชาวประมงไทยผลิตทดแทน (Import Substitution)

# 🦐 Step 3: วิเคราะห์โครงสร้างสินค้าประมง (Fisheries Product Structure)
**🎯 โจทย์ข้อ 2:** จัดกลุ่มตาม HS Code เพื่อหาว่าสินค้าประเภทใดที่ไทยเป็นผู้ผลิตส่งออกสุทธิ (Net Export) และประเภทใดต้องพึ่งพาการนำเข้า (Net Import)

In [8]:
# ============================================================
# 🦐 Step 3: โครงสร้างสินค้าประมงไทย (Fisheries Product Structure)
# ============================================================

# 1. เตรียมข้อมูล: จัดกลุ่มตามรหัสสินค้า (heading11) และชื่อสินค้า (productDetailTH)
product_trade = df_clean.groupby(['heading11', 'productDetailTH', 'trade_type'])['price_actual'].sum().unstack(fill_value=0).reset_index()

if 'Import' not in product_trade.columns: product_trade['Import'] = 0
if 'Export' not in product_trade.columns: product_trade['Export'] = 0

# คำนวณดุลการค้า (Net Balance)
product_trade['Net Balance'] = product_trade['Export'] - product_trade['Import']

# เรียงลำดับจาก "ขาดดุลมากสุด (Net Import)" ไปหา "กำไรมากสุด (Net Export)" เพื่อให้กราฟอ่านง่าย
product_trade = product_trade.sort_values(by='Net Balance', ascending=True)

# 2. วาดกราฟ
fig_q2 = go.Figure()

# ฝั่งส่งออก (สีฟ้า)
fig_q2.add_trace(go.Bar(
    y=product_trade['productDetailTH'],
    x=product_trade['Export'],
    name='ส่งออก (Export)',
    orientation='h',
    marker_color=Theme.export_col,
    hovertemplate="<b>%{y}</b><br>HS Code: %{customdata}<br>ส่งออก: ฿%{x:,.0f}<extra></extra>",
    customdata=product_trade['heading11'] # แนบ HS Code ไว้โชว์ตอน Hover
))

# ฝั่งนำเข้า (สีแดง - ชี้ไปทางซ้าย)
fig_q2.add_trace(go.Bar(
    y=product_trade['productDetailTH'],
    x=-product_trade['Import'],
    name='นำเข้า (Import)',
    orientation='h',
    marker_color=Theme.import_col,
    hovertemplate="<b>%{y}</b><br>HS Code: %{text}<br>นำเข้า: ฿%{customdata:,.0f}<extra></extra>",
    customdata=product_trade['Import'],
    text=product_trade['heading11']
))

# จุดแสดงสถานะ (Net Balance)
fig_q2.add_trace(go.Scatter(
    y=product_trade['productDetailTH'],
    x=product_trade['Net Balance'],
    name='สถานะสุทธิ (Net Exporter / Importer)',
    mode='markers',
    marker=dict(color=Theme.accent, size=12, symbol='circle'),
    hovertemplate="<b>%{y}</b><br>สถานะสุทธิ: ฿%{x:,.0f}<extra></extra>"
))

# 3. จัด Layout
fig_q2.update_layout(
    title=dict(text="<b>🦐 โครงสร้างสินค้าประมงแยกตาม HS Code (Net Export vs Net Import)</b>", font=dict(size=20), x=0.5),
    barmode='overlay',
    height=650, # เผื่อพื้นที่ให้ชื่อสินค้ายาวๆ
    xaxis=dict(
        title='มูลค่าการค้า (บาท)',
        zeroline=True, zerolinewidth=2, zerolinecolor=Theme.text
    ),
    yaxis=dict(
        title='ประเภทสินค้าประมง',
        tickfont=dict(size=11),
        automargin=True # ป้องกันชื่อสินค้าโดนตัดตกขอบ
    ),
    hovermode="y unified"
)

fig_q2.show()

### 📌 Insight & So What? (สรุปผลข้อ 2)
* 🌟 **กลุ่มที่เป็นหัวใจการส่งออก (Net Exporters):** สินค้าที่จุดสีส้มอยู่ทางฝั่งขวาบนสุดคือกำลังหลักของประเทศ (เช่น *รอใส่ชื่อสินค้าจริงที่ได้จากกราฟ*) ไทยมีสถานะเป็น "ผู้ผลิต" ที่แข่งขันได้ในตลาดโลก
* ⚠️ **กลุ่มที่พึ่งพาการนำเข้า (Net Importers):** สินค้าที่จุดสีส้มตกไปอยู่ฝั่งซ้ายล่างสุด แสดงว่าไทยไม่สามารถผลิตได้เพียงพอ หรือต้องนำเข้าวัตถุดิบราคาถูกเข้ามาเพื่อป้อนโรงงานแปรรูป (เช่น *รอใส่ชื่อสินค้าจริงที่ได้จากกราฟ*)
* 🚀 **So What?:** * รัฐบาลควรจัดสรรงบวิจัย (R&D) เพื่อเพิ่มมูลค่า (Value-added) ให้กับกลุ่ม Net Exporters เพื่อหนีการตัดราคาจากประเทศเพื่อนบ้าน
  * สำหรับกลุ่ม Net Importers หากเป็นวัตถุดิบต้นน้ำ ควรเจรจาลดภาษีนำเข้าเพื่อช่วยลดต้นทุนให้โรงงานแปรรูปอาหารทะเลในไทย

# 📈 Step 4: วิเคราะห์ฤดูกาลการค้า (Seasonal Trade Pattern)
**🎯 โจทย์ข้อ 3:** วิเคราะห์แนวโน้มรายเดือนเพื่อหาจุดสูงสุด (Peak) และจุดต่ำสุด (Low) ของการค้า พร้อมตั้งสมมติฐานถึงปัจจัยที่ส่งผลกระทบ


In [9]:
# ============================================================
# 📈 Step 4: แนวโน้มฤดูกาลการค้าสัตว์น้ำ (Seasonal Trade Pattern)
# ============================================================

# 1. เตรียมข้อมูล: จัดกลุ่มมูลค่าการค้าตามเดือนและฝั่งการค้า
monthly_trade = df_clean.groupby(['month', 'trade_type'])['price_actual'].sum().unstack(fill_value=0).reset_index()

# สร้างชื่อเดือนให้เป็นภาษาอังกฤษ (Jan, Feb, ...) เพื่อให้อ่านง่ายกว่าตัวเลข 1-12
monthly_trade['month_name'] = monthly_trade['month'].apply(lambda x: calendar.month_abbr[x])

# หาเดือนที่เป็น Peak และ Low ของฝั่งส่งออก (Export) เพื่อทำ Label อัตโนมัติ
max_export_idx = monthly_trade['Export'].idxmax()
min_export_idx = monthly_trade['Export'].idxmin()

max_month = monthly_trade.loc[max_export_idx, 'month_name']
max_val = monthly_trade.loc[max_export_idx, 'Export']
min_month = monthly_trade.loc[min_export_idx, 'month_name']
min_val = monthly_trade.loc[min_export_idx, 'Export']

# 2. วาดกราฟเส้น (Line Chart)
fig_q3 = go.Figure()

# เส้นส่งออก (Export)
fig_q3.add_trace(go.Scatter(
    x=monthly_trade['month_name'],
    y=monthly_trade['Export'],
    name='ส่งออก (Export)',
    mode='lines+markers',
    line=dict(color=Theme.export_col, width=4),
    marker=dict(size=8, color=Theme.bg_dark, line=dict(width=2, color=Theme.export_col)),
    hovertemplate="<b>%{x}</b><br>ส่งออก: ฿%{y:,.0f}<extra></extra>"
))

# เส้นนำเข้า (Import)
fig_q3.add_trace(go.Scatter(
    x=monthly_trade['month_name'],
    y=monthly_trade['Import'],
    name='นำเข้า (Import)',
    mode='lines+markers',
    line=dict(color=Theme.import_col, width=3, dash='dot'), # ทำให้เส้นนำเข้าดูเบากว่า (เป็นเส้นประ)
    marker=dict(size=6),
    hovertemplate="<b>%{x}</b><br>นำเข้า: ฿%{y:,.0f}<extra></extra>"
))

# 3. ไฮไลต์จุด Peak และ Low ด้วย Annotation
fig_q3.add_annotation(
    x=max_month, y=max_val,
    text=f"<b>🌟 Peak Season</b><br>(สูงสุดในรอบปี)",
    showarrow=True, arrowhead=2, arrowsize=1.5, arrowcolor=Theme.accent,
    font=dict(size=12, color=Theme.accent),
    ax=0, ay=-45
)

fig_q3.add_annotation(
    x=min_month, y=min_val,
    text=f"<b>⚠️ Low Season</b><br>(ต่ำสุดในรอบปี)",
    showarrow=True, arrowhead=2, arrowcolor=Theme.text,
    font=dict(size=11, color=Theme.text),
    ax=0, ay=45
)

# 4. จัด Layout
fig_q3.update_layout(
    title=dict(text="<b>📈 วัฏจักรการค้าสัตว์น้ำรายเดือน (Seasonality Trend)</b>", font=dict(size=20), x=0.5),
    height=500,
    xaxis=dict(title='เดือน (Month)', showgrid=False),
    yaxis=dict(title='มูลค่าการค้า (บาท)', zeroline=True, zerolinewidth=1, zerolinecolor=Theme.grid),
    hovermode="x unified"
)

fig_q3.show()

### 📌 Insight & So What? (สมมติฐานและบทสรุปข้อ 3)
* 📈 **Peak Season (ช่วงพีกสุด):** กราฟแสดงยอดส่งออกพุ่งสูงสุดในช่วง **[เดือน.... ถึง เดือน....]** * *สมมติฐาน:* เกิดจากความต้องการของตลาดโลกที่เร่งสต็อกสินค้าประมงแช่แข็งและแปรรูป เพื่อเตรียมรองรับเทศกาลสิ้นปี (Christmas & New Year) รวมถึงเป็นช่วงที่ผลผลิตจากการเพาะเลี้ยง (เช่น กุ้งขาว) ออกสู่ตลาดจำนวนมาก
* 📉 **Low Season (ช่วงต่ำสุด):** ยอดเทรดดรอปลงอย่างเห็นได้ชัดในเดือน **[เดือน....]** * *สมมติฐาน:* อาจตรงกับช่วงฤดูมรสุมของไทย (พฤษภาคม - ตุลาคม) ที่เรือประมงพาณิชย์ออกหาปลาได้น้อยลง หรือเป็นช่วงประกาศปิดอ่าวไทยเพื่อฟื้นฟูทรัพยากร (ทะเลพักฟื้น) ทำให้วัตถุดิบขาดแคลนชั่วคราว
* 🚀 **So What?:** * **การจัดการ Supply Chain:** โรงงานแปรรูปควรเร่งนำเข้าวัตถุดิบมาตุนไว้ (Buffer Stock) ก่อนเข้าสู่ช่วงฤดูมรสุม เพื่อไม่ให้สายการผลิตสะดุด
   * **กลยุทธ์การตลาด:** ภาครัฐควรสนับสนุนระบบห้องเย็น (Cold Storage) ที่มีประสิทธิภาพ เพื่อให้เกษตรกรสามารถเก็บรักษาสัตว์น้ำในช่วง Low Season และนำมาเก็งกำไรขายในช่วง Peak Season ที่ทั่วโลกมีความต้องการสูง